# Rice Leaf Disease Classification — PRCP-1001

**Single-notebook capstone:** dataset audit and EDA, image preprocessing, augmentation experiments, model training/comparison, evaluation, error analysis, and a production recommendation.

**Dataset supplied:** `PRCP-1001-RiceLeaf.zip` (contains three class-specific ZIP files). The notebook extracts these nested archives automatically. The project brief lists **Leaf smut, Brown spot, Bacterial leaf blight**. The supplied archive contains 39 Leaf smut images and 40 images in each other class (119 total), so the notebook reports the observed counts rather than assuming 120.

> Run cells from top to bottom. A GPU is helpful but not required. The first run may download ImageNet weights for MobileNetV2; set `USE_PRETRAINED=False` if offline.


## 1. Environment and imports

In [ ]:
# If an import fails in a fresh environment, uncomment and run:
# %pip install -q numpy pandas matplotlib seaborn pillow scikit-learn scikit-image tensorflow joblib

import os, sys, zipfile, shutil, random, warnings, json, time
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print('TensorFlow:', tf.__version__)
except ImportError as e:
    raise ImportError('TensorFlow is required. Run: %pip install tensorflow') from e

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_recall_fscore_support)
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from skimage.feature import hog
import joblib
print('Working directory:', Path.cwd())
print('Available GPUs:', tf.config.list_physical_devices('GPU'))

## 2. Dataset path and extraction

The primary path below matches the uploaded dataset location in this environment. If you move the notebook, place the ZIP beside it or update `DATASET_ZIP`.

In [ ]:
# PRIMARY PATH: uploaded dataset ZIP
DATASET_ZIP = Path('/mnt/data/PRCP-1001-RiceLeaf.zip')
# Portable fallback: ZIP in the same folder as this notebook
if not DATASET_ZIP.exists():
    DATASET_ZIP = Path.cwd() / 'PRCP-1001-RiceLeaf.zip'

EXTRACT_DIR = Path.cwd() / 'riceleaf_data'
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
assert DATASET_ZIP.exists(), f'Dataset ZIP not found: {DATASET_ZIP}. Update DATASET_ZIP to the correct path.'

# Outer ZIP contains one ZIP per disease class; extract all image files into class folders.
with zipfile.ZipFile(DATASET_ZIP, 'r') as outer:
    nested_archives = [n for n in outer.namelist() if n.lower().endswith('.zip')]
    print('Nested archives:', nested_archives)
    for nested_name in nested_archives:
        class_name = Path(nested_name).name.split('-2020')[0]
        class_dir = EXTRACT_DIR / class_name
        class_dir.mkdir(parents=True, exist_ok=True)
        nested_bytes = outer.read(nested_name)
        import io
        with zipfile.ZipFile(io.BytesIO(nested_bytes)) as inner:
            for member in inner.namelist():
                if member.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                    target = class_dir / Path(member).name
                    if not target.exists():
                        target.write_bytes(inner.read(member))

print('Extracted to:', EXTRACT_DIR.resolve())
for d in sorted(EXTRACT_DIR.iterdir()):
    if d.is_dir(): print(f'{d.name}: {len(list(d.glob("*")))} files')

## 3. Build image index and audit the data

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
records = []
for class_dir in sorted(p for p in EXTRACT_DIR.iterdir() if p.is_dir()):
    for path in sorted(class_dir.iterdir()):
        if path.suffix.lower() in IMAGE_EXTS:
            records.append({'filepath': str(path), 'label': class_dir.name, 'filename': path.name})
df = pd.DataFrame(records)
assert not df.empty, 'No images found. Check the extraction and dataset path.'
print('Total images:', len(df))
print('Classes:', sorted(df.label.unique()))
print('Class counts:', df.label.value_counts())
print('Missing file paths:', (~df.filepath.map(lambda p: Path(p).exists())).sum())

# Verify that images can be opened and record dimensions / modes.
image_info, corrupt = [], []
for row in df.itertuples():
    try:
        with Image.open(row.filepath) as im:
            image_info.append({'filepath': row.filepath, 'width': im.width, 'height': im.height,
                               'mode': im.mode, 'format': im.format})
    except Exception as e:
        corrupt.append((row.filepath, str(e)))
info = pd.DataFrame(image_info)
print(f'Readable images: {len(info)} | Unreadable images: {len(corrupt)}')
if corrupt: print('Unreadable examples:', corrupt[:5])
df = df[df.filepath.isin(info.filepath)].reset_index(drop=True)
df = df.merge(info, on='filepath', how='left')
display(df.head())

## 4. Exploratory data analysis (EDA)

In [ ]:
plt.figure(figsize=(8,4))
order = df.label.value_counts().index
sns.countplot(data=df, x='label', order=order)
plt.title('Images per disease class'); plt.xlabel('Disease class'); plt.ylabel('Image count')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

display(df.groupby('label')[['width','height']].agg(['min','median','max']))
fig, axes = plt.subplots(1, 3, figsize=(13,4))
for ax, (label, group) in zip(axes, df.groupby('label')):
    ax.hist(group['width'], alpha=.7, label='width')
    ax.hist(group['height'], alpha=.7, label='height')
    ax.set_title(label); ax.set_xlabel('Pixels'); ax.legend()
plt.tight_layout(); plt.show()

# Display up to four examples per class.
classes = sorted(df.label.unique())
fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3*len(classes)))
for r, label in enumerate(classes):
    sample = df[df.label == label].sample(min(4, (df.label == label).sum()), random_state=SEED)
    for c, ax in enumerate(axes[r]):
        ax.axis('off')
        if c < len(sample):
            row = sample.iloc[c]
            with Image.open(row.filepath) as im: ax.imshow(im.convert('RGB'))
            ax.set_title(label if c == 0 else row.filename, fontsize=8)
plt.tight_layout(); plt.show()

## 5. Split data before augmentation

Use a stratified train/validation/test split. Augmentation is applied only to training batches, preventing augmented variants of an image from leaking into validation or test data.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df.label, random_state=SEED)
class_names = sorted(df.label.unique())
class_to_id = {name:i for i,name in enumerate(class_names)}
NUM_CLASSES = len(class_names)
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

def make_dataset(frame, training=False):
    paths = frame.filepath.to_numpy()
    labels = frame.label.map(class_to_id).to_numpy(dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32)
        return image, label
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
print('Train:', len(train_df), 'Validation:', len(val_df), 'Test:', len(test_df))
for name, part in [('train',train_df),('validation',val_df),('test',test_df)]:
    print(name, part.label.value_counts().to_dict())

## 6. Data augmentation experiment

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name='training_augmentation')

# Visualize original and augmented versions of one training image.
for images, labels in train_ds.take(1):
    sample = images[:1]
    augmented = data_augmentation(tf.repeat(sample, 5, axis=0), training=True)
    fig, axes = plt.subplots(1, 6, figsize=(15,3))
    axes[0].imshow(sample[0].numpy().astype('uint8')); axes[0].set_title('Original')
    for i in range(5): axes[i+1].imshow(tf.clip_by_value(augmented[i],0,255).numpy().astype('uint8')); axes[i+1].set_title(f'Augmented {i+1}')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()
    break

## 7. Model 1 — CNN trained from scratch

In [ ]:
def build_custom_cnn():
    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./255)(x)
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(0.20)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.35)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = keras.Model(inputs, outputs, name='custom_cnn')
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
             keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6)]
EPOCHS = 25
cnn_model = build_custom_cnn()
cnn_history = cnn_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)

## 8. Model 2 — MobileNetV2 transfer learning

In [ ]:
# ImageNet pretrained features are useful when the dataset is small.
USE_PRETRAINED = True  # Set False if the environment has no internet / cached ImageNet weights.
weights = 'imagenet' if USE_PRETRAINED else None
base = keras.applications.MobileNetV2(input_shape=(*IMG_SIZE,3), include_top=False, weights=weights)
base.trainable = False
inputs = keras.Input(shape=(*IMG_SIZE,3))
x = data_augmentation(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.30)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
transfer_model = keras.Model(inputs, outputs, name='mobilenetv2_transfer')
transfer_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])
transfer_history = transfer_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=1)

# Optional fine-tuning: unfreeze only the final layers and use a small learning rate.
base.trainable = True
for layer in base.layers[:-30]: layer.trainable = False
transfer_model.compile(optimizer=keras.optimizers.Adam(1e-5),
                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])
finetune_history = transfer_model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks, verbose=1)

## 9. Model 3 — HOG features + SVM baseline

A traditional machine-learning baseline provides a useful comparison against deep learning. HOG captures local edge/texture structure; the SVM is trained only on the training split.

In [ ]:
HOG_SIZE = (128,128)
def extract_hog(path):
    with Image.open(path) as im:
        im = im.convert('RGB').resize(HOG_SIZE)
        arr = np.asarray(im)
    return hog(arr, orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2),
               channel_axis=-1, block_norm='L2-Hys')

X_train_hog = np.vstack([extract_hog(p) for p in train_df.filepath])
y_train = train_df.label.to_numpy()
X_val_hog = np.vstack([extract_hog(p) for p in val_df.filepath])
X_test_hog = np.vstack([extract_hog(p) for p in test_df.filepath])
hog_svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=3, probability=True, class_weight='balanced', random_state=SEED))
hog_svm.fit(X_train_hog, y_train)
print('HOG-SVM validation accuracy:', accuracy_score(val_df.label, hog_svm.predict(X_val_hog)))

## 10. Training curves

In [ ]:
def plot_history(history, title):
    h = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1,2,figsize=(11,4))
    axes[0].plot(h['loss'], label='train'); axes[0].plot(h['val_loss'], label='validation')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(h['accuracy'], label='train'); axes[1].plot(h['val_accuracy'], label='validation')
    axes[1].set_title('Accuracy'); axes[1].legend()
    fig.suptitle(title); plt.tight_layout(); plt.show()
plot_history(cnn_history, 'Custom CNN')
plot_history(transfer_history, 'MobileNetV2 — frozen backbone')
plot_history(finetune_history, 'MobileNetV2 — fine tuning')

## 11. Test-set evaluation and model comparison

The test set is held out from fitting. Evaluate all candidates on the same test images and compare accuracy, macro precision, macro recall, and macro F1. For small/imbalanced datasets, macro F1 and class-wise recall are especially important.

In [ ]:
def collect_true_and_pred(model, dataset):
    y_true, y_prob = [], []
    for images, labels in dataset:
        probs = model.predict(images, verbose=0)
        y_true.extend(labels.numpy().tolist()); y_prob.extend(probs.tolist())
    y_prob = np.asarray(y_prob)
    return np.asarray(y_true), y_prob.argmax(axis=1), y_prob

def score_predictions(name, y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return {'Model':name, 'Accuracy':accuracy_score(y_true,y_pred),
            'Macro precision':p, 'Macro recall':r, 'Macro F1':f1}

results = []
model_outputs = {}
for name, model in [('Custom CNN',cnn_model), ('MobileNetV2',transfer_model)]:
    yt, yp, probs = collect_true_and_pred(model, test_ds)
    model_outputs[name] = (yt,yp,probs)
    results.append(score_predictions(name, yt, yp))

svm_pred = hog_svm.predict(X_test_hog)
svm_true = test_df.label.map(class_to_id).to_numpy()
# Convert SVM labels to the same integer class encoding.
svm_pred_ids = np.array([class_to_id[x] for x in svm_pred])
results.append(score_predictions('HOG + SVM', svm_true, svm_pred_ids))
comparison = pd.DataFrame(results).sort_values('Macro F1', ascending=False).reset_index(drop=True)
display(comparison.style.format({c:'{:.3f}' for c in comparison.columns if c != 'Model'}))
best_model_name = comparison.iloc[0]['Model']
print('Best observed model by test Macro F1:', best_model_name)

## 12. Confusion matrices and class-wise report

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
for ax, (name, (yt,yp,_)) in zip(axes[:2], model_outputs.items()):
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.tick_params(axis='x', rotation=35)
cm = confusion_matrix(svm_true, svm_pred_ids, labels=list(range(NUM_CLASSES)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[2])
axes[2].set_title('HOG + SVM'); axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual'); axes[2].tick_params(axis='x', rotation=35)
plt.tight_layout(); plt.show()

for name, (yt,yp,_) in model_outputs.items():
    print('\n', name)
    print(classification_report(yt, yp, labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0))
print('\nHOG + SVM')
print(classification_report(svm_true, svm_pred_ids, labels=list(range(NUM_CLASSES)), target_names=class_names, zero_division=0))

## 13. Error analysis — inspect incorrect predictions

In [ ]:
# Select the highest-ranked deep-learning model for visual error inspection.
# HOG-SVM is included in the comparison, but this cell visualizes the best neural network.
chosen_name = 'MobileNetV2' if comparison.iloc[0]['Model'] == 'MobileNetV2' else 'Custom CNN'
chosen_model = transfer_model if chosen_name == 'MobileNetV2' else cnn_model
misclassified = []
for row in test_df.itertuples():
    with Image.open(row.filepath) as im:
        arr = np.asarray(im.convert('RGB').resize(IMG_SIZE)).astype('float32')
    probs = chosen_model.predict(arr[None,...], verbose=0)[0]
    pred = class_names[int(np.argmax(probs))]
    if pred != row.label:
        misclassified.append((row.filepath,row.label,pred,float(np.max(probs))))
print(f'{len(misclassified)} misclassified out of {len(test_df)} test images using {chosen_name}.')
if misclassified:
    n = min(9,len(misclassified)); fig, axes = plt.subplots((n+2)//3,3,figsize=(12,4*((n+2)//3)))
    for ax,item in zip(np.array(axes).reshape(-1),misclassified[:n]):
        with Image.open(item[0]) as im: ax.imshow(im.convert('RGB'))
        ax.set_title(f'True: {item[1]}\nPred: {item[2]} ({item[3]:.2f})',fontsize=9); ax.axis('off')
    for ax in np.array(axes).reshape(-1)[n:]: ax.axis('off')
    plt.tight_layout(); plt.show()

## 14. Data augmentation impact (controlled experiment)

The CNN and transfer-learning models above include training-only augmentation. The following optional ablation trains a smaller CNN **without augmentation** to provide a direct reference. Results can vary because this dataset is very small; interpret differences alongside validation curves and class-wise metrics.

In [ ]:
def build_cnn_without_augmentation():
    inputs = keras.Input(shape=(*IMG_SIZE,3))
    x = layers.Rescaling(1./255)(inputs)
    for filters in [32,64,128]:
        x = layers.Conv2D(filters,3,padding='same',activation='relu')(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(.2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128,activation='relu')(x)
    x = layers.Dropout(.3)(x)
    outputs = layers.Dense(NUM_CLASSES,activation='softmax')(x)
    m = keras.Model(inputs,outputs,name='cnn_without_augmentation')
    m.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

RUN_AUGMENTATION_ABLATION = True
if RUN_AUGMENTATION_ABLATION:
    no_aug_model = build_cnn_without_augmentation()
    no_aug_history = no_aug_model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks, verbose=1)
    yt,yp,_ = collect_true_and_pred(no_aug_model,test_ds)
    aug_ablation = pd.DataFrame([
        score_predictions('CNN with augmentation', *model_outputs.get('Custom CNN', (yt,yp,None))[:2]),
        score_predictions('CNN without augmentation',yt,yp)
    ])
    display(aug_ablation)
    plot_history(no_aug_history,'CNN without augmentation')

## 15. Save the selected model and class mapping

In [ ]:
# Select the top-performing candidate from the held-out comparison.
# Save neural network as .keras; save SVM pipeline with joblib.
OUTPUT_DIR = Path.cwd() / 'riceleaf_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
if best_model_name == 'MobileNetV2':
    transfer_model.save(OUTPUT_DIR / 'riceleaf_mobilenetv2.keras')
elif best_model_name == 'Custom CNN':
    cnn_model.save(OUTPUT_DIR / 'riceleaf_custom_cnn.keras')
else:
    joblib.dump(hog_svm, OUTPUT_DIR / 'riceleaf_hog_svm.joblib')

with open(OUTPUT_DIR / 'class_mapping.json','w') as f:
    json.dump({'class_names':class_names,'class_to_id':class_to_id},f,indent=2)
comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv',index=False)
print('Saved outputs to:', OUTPUT_DIR.resolve())
print('Selected candidate:', best_model_name)

## 16. Single-image prediction helper

In [ ]:
def predict_rice_leaf(image_path, model=None):
    """Predict disease class for one image using the selected neural network."""
    if model is None:
        if best_model_name == 'MobileNetV2': model = transfer_model
        elif best_model_name == 'Custom CNN': model = cnn_model
        else: raise ValueError('The selected model is HOG-SVM; use predict_rice_leaf_svm for this model.')
    with Image.open(image_path) as im:
        image = np.asarray(im.convert('RGB').resize(IMG_SIZE)).astype('float32')
    probs = model.predict(image[None,...],verbose=0)[0]
    order = np.argsort(probs)[::-1]
    return pd.DataFrame({'Disease': [class_names[i] for i in order], 'Probability': [float(probs[i]) for i in order]})

def predict_rice_leaf_svm(image_path):
    features = extract_hog(image_path).reshape(1,-1)
    probs = hog_svm.predict_proba(features)[0]
    order = np.argsort(probs)[::-1]
    return pd.DataFrame({'Disease':[hog_svm.classes_[i] for i in order], 'Probability':[float(probs[i]) for i in order]})

# Example: predict_rice_leaf(df.iloc[0]['filepath'])
# Or, if HOG-SVM is selected: predict_rice_leaf_svm(df.iloc[0]['filepath'])

## 17. Findings, challenges, and production recommendation

### Dataset findings (computed from the supplied archive)
- Three disease classes are present: Leaf smut, Brown spot, and Bacterial leaf blight.
- The supplied archive contains 39 Leaf smut images, 40 Brown spot images, and 40 Bacterial leaf blight images (119 total), rather than the 120 stated in the brief.
- This is a small image dataset. Metrics can vary substantially with the split; results should not be treated as production-grade validation.

### Challenges and techniques used
1. **Small dataset:** use transfer learning and moderate augmentation to reduce overfitting.
2. **Unequal class counts:** use stratified splits and macro-averaged metrics; avoid assuming perfectly equal classes.
3. **Different image dimensions / formats:** decode images consistently as RGB and resize to 224×224.
4. **Potential image memorization / leakage:** split original images before applying random augmentation; augmentation occurs only on training batches.
5. **Limited test sample size:** report confusion matrices and class-wise precision/recall, and collect more independently sourced field images before deployment.
6. **Field domain shift:** lighting, backgrounds, camera quality, leaf age, and disease severity may differ from curated images; test on real farm images.

### Production recommendation
Use the candidate with the strongest held-out **macro F1**, while checking that no disease class has unacceptably low recall. MobileNetV2 transfer learning is a sensible deployment candidate for a small image dataset, but the notebook selects the best measured model rather than hard-coding a winner. Before production, collect a larger, diverse, independently labelled dataset; use a farm/location-separated test set; calibrate confidence; add an 'uncertain / refer to expert' threshold; and monitor performance after deployment.

### Final result summary (run this cell after all training/evaluation cells)

In [ ]:
print('Dataset size:', len(df))
print('Class counts:', df.label.value_counts().to_dict())
print('Train/validation/test:', len(train_df), len(val_df), len(test_df))
print('Model ranking:')
display(comparison)
print('Recommended candidate based on this split:', best_model_name)
print('Important: with only 119 images, obtain external validation before any real-world diagnosis or treatment decisions.')

---
**Submission note:** This notebook implements the project tasks in one Jupyter notebook. It does not claim a fixed accuracy in advance; the actual scores, charts, and model recommendation are generated when you execute the training cells on your machine.